<a href="https://colab.research.google.com/github/sllamayo/ap155_project_euler/blob/main/Problem_Set_1_sllamayo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Problem 4.17

import numpy as np
import warnings
warnings.filterwarnings("ignore")

#Backward and Forward sub:
def forward_sub(L, b):
    n = L.shape[0]
    y = np.zeros_like(b, dtype = float)
    for i in range(n):
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i,i]
    return y

def backward_sub(U, y):
    n = U.shape[0]
    x = np.zeros_like(y, dtype = float)
    for i in reversed(range(n)):
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x

#LU Decomposition:
def ludec(A):
 n = A.shape[0]
 U = A.copy().astype(float)
 L = np.identity(n)

 for j in range(n-1):
  for i in range(j+1,n):
    coeff = U[i,j]/U[j,j]
    U[i,j:]-= coeff*U[j,j:]
    L[i,j] = coeff
 return L, U


#Pivoting
def partial_pivot(A, b):
    n = A.shape[0]
    P = np.eye(n)
    A = A.copy()
    b = b.copy()
    for k in range(n-1):
        pivot = np.argmax(abs(A[k:, k])) + k
        if pivot != k:
            A[[k, pivot], :] = A[[pivot, k], :]
            b[[k, pivot]] = b[[pivot, k]]
            P[[k, pivot], :] = P[[pivot, k], :]
    return P, A, b


#LU Decomposition but with 10^-20 replacement

def ludec_safe(A):
 n = A.shape[0]
 U = A.copy().astype(float)
 L = np.identity(n)
 small = 1e-20

 for j in range(n-1):
  for i in range(j+1,n):
    coeff = U[i,j]/U[j,j]
    U[i,j:]-= coeff*U[j,j:]
    L[i,j] = coeff

  if U[n-1,n-1] == 0:
    U[n-1,n-1] = small

 return L, U

#Given

A= np.array([4., 4, 8,4, 4, 5,3, 7, 8, \
 3, 9,9, 4, 7,9, 5]).reshape(4,4)

bs = np.array([1., 2, 3, 4])


#Without pivot
L_no_pivot,U_no_pivot = ludec(A)
y_no_pivot = forward_sub(L_no_pivot, bs)
x_no_pivot = backward_sub(U_no_pivot, y_no_pivot)
print("without pivot:\n", x_no_pivot)

#With pivot
P, A_pivoted, bs_pivoted = partial_pivot(A, bs)
L_pivot, U_pivot = ludec(A_pivoted)
y_pivoted = forward_sub(L_pivot, bs_pivoted)
x_pivoted = backward_sub(U_pivot, y_pivoted)
print("with pivot:\n ", x_pivoted)

print("\nPivoting did not help in this case since an element in U matrix (U_33) for both methods is zero.\n")

#Without pivot but with 10^-20 replacement
L_no_pivot_safe,U_no_pivot_safe = ludec_safe(A)
y_no_pivot_safe = forward_sub(L_no_pivot_safe, bs)
x_no_pivot_safe = backward_sub(U_no_pivot_safe, y_no_pivot_safe)
print("without pivot but with replacement:\n ", x_no_pivot_safe)

#With pivot but with 10^-20 replacement
L_pivot_safe,U_pivot_safe = ludec_safe(A_pivoted)
y_pivot_safe = forward_sub(L_pivot_safe, bs)
x_pivot_safe = backward_sub(U_pivot_safe, y_pivot_safe)
print("without pivot but with replacement:\n", x_pivot_safe)

print("\nReplacing the U_33 element with a very tiny number did not help in this case since we are dividing by a very small number which results to a big number.\n")



without pivot:
 [nan nan inf inf]
with pivot:
  [ nan  inf -inf -inf]

Pivoting did not help in this case since an element in U matrix (U_33) for both methods is zero.

without pivot but with replacement:
  [-4.5e+20 -1.5e+20  1.5e+20  3.0e+20]
without pivot but with replacement:
 [-1.2e+21 -4.0e+20  4.0e+20  8.0e+20]

Replacing the U_33 element with a very tiny number did not help in this case since we are dividing by a very small number which results to a big number.



In [ ]:
#Problem 4.18

import numpy as np

#jacobi cyclic tridiagonal generator

def jacobi_tridiag (n, tol = 1e-5, max_iter=10000):

  x_old = np.zeros(n)
  x_new = np.zeros(n)

  for k in range(max_iter):
    x_new[0] = (1 + x_old[1] + x_old[-1])/4.0

    for i in range(1, n-1):
      x_new[i] = (1+ x_old[i-1] + x_old[i+1])/4.0

    x_new[-1] = (1 + x_old[0] + x_old[-2])/4.0

    error = np.max(np.abs(x_new - x_old))
    if error < tol:
       print(f"Jacobi iterations converged after {k+1} iterations")
       return x_new

    x_old[:] = x_new[:]

  print("Did not converge within max iterations")
  return x_new

#building the matrix

def A_jacobi(n):
  A = np.zeros((n,n))
  np.fill_diagonal(A, 4)

  for i in range(n):
    if i > 0:
      A[i, i-1] = -1
    if i < n-1:
      A[i, i+1] = -1

  A[0,-1] = -1
  A[-1, 0] = -1
  return A


# for n = 10
print("For n = 10")
A_10 = A_jacobi(10)
b_10 = np.ones(10)
numpy_10 = np.linalg.solve(A_10,b_10)
jacobi_10 = jacobi_tridiag(n=10, tol=1e-5)
print("\nUsing jacobi iterations:")
print(jacobi_10)
print("\nUsing numpy:")
print(numpy_10)
print("\nDifference")
print(abs(jacobi_10-numpy_10))


#for n = 20
print("\nFor n = 20:")
A_20 = A_jacobi(20)
b_20 = np.ones(20)
numpy_20 = np.linalg.solve(A_20,b_20)
jacobi_20 = jacobi_tridiag(n=20, tol=1e-5)
print("\nUsing jacobi iterations:")
print(jacobi_20)
print("\nUsing numpy:")
print(numpy_20)
print("\nDifference")
print(abs(jacobi_20-numpy_20))



For n = 10
Jacobi iterations converged after 16 iterations

Using jacobi iterations:
[0.49999237 0.49999237 0.49999237 0.49999237 0.49999237 0.49999237
 0.49999237 0.49999237 0.49999237 0.49999237]

Using numpy:
[0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]

Difference
[7.62939453e-06 7.62939453e-06 7.62939453e-06 7.62939453e-06
 7.62939453e-06 7.62939453e-06 7.62939453e-06 7.62939453e-06
 7.62939453e-06 7.62939453e-06]

For n = 20:
Jacobi iterations converged after 16 iterations

Using jacobi iterations:
[0.49999237 0.49999237 0.49999237 0.49999237 0.49999237 0.49999237
 0.49999237 0.49999237 0.49999237 0.49999237 0.49999237 0.49999237
 0.49999237 0.49999237 0.49999237 0.49999237 0.49999237 0.49999237
 0.49999237 0.49999237]

Using numpy:
[0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5
 0.5 0.5]

Difference
[7.62939453e-06 7.62939453e-06 7.62939453e-06 7.62939453e-06
 7.62939453e-06 7.62939453e-06 7.62939453e-06 7.62939453e-06
 7.62939453e-06 7.62939453e-06 7.6293